# Exploratory Data Analysis (EDA)

In this notebook, we pull the clean data from our SQLite database and analyze long-term trends, volatility, and historical price cycles for our commodities.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load data from DB
conn = sqlite3.connect('../sql/commodity_data.db')

query = """
SELECT 
    p.date, 
    c.commodity_name, 
    c.category, 
    p.price_nominal_usd, 
    p.price_mom_pct, 
    p.price_12m_volatility
FROM prices p 
JOIN commodities c ON p.commodity_id = c.commodity_id
"""

df = pd.read_sql_query(query, conn)
df['date'] = pd.to_datetime(df['date'])
conn.close()

df.head()

## 1. Price Evolution of Key Commodities
Let's look at Gold, Crude oil, and Wheat as representatives of Metals, Energy, and Agriculture.

In [ ]:
key_commodities = ['Gold', 'Crude oil, average', 'Wheat, US SRW']
subs = df[df['commodity_name'].isin(key_commodities)]

g = sns.FacetGrid(subs, col="commodity_name", col_wrap=3, sharey=False, height=4, aspect=1.5)
g.map(sns.lineplot, "date", "price_nominal_usd")
g.set_axis_labels("Year", "Price (USD)")
g.set_titles(col_template="{col_name}")
plt.suptitle('Historical Price Evolution', y=1.05, fontsize=16)
plt.savefig('../images/historical_prices.png', bbox_inches='tight')
plt.show()

## 2. Volatility Analysis
Let's see which category of commodities is the most volatile.

In [ ]:
# Average 12m volatility by category
vol_cat = df.groupby('category')['price_12m_volatility'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=vol_cat, x='price_12m_volatility', y='category', hue='category', palette="viridis", dodge=False, legend=False)
plt.title('Average 12-Month Volatility by Commodity Category')
plt.xlabel('Average Volatility')
plt.ylabel('Category')
plt.savefig('../images/volatility_by_category.png', bbox_inches='tight')
plt.show()